# Prueba técnica Creceré AI — Etapa 3: variables de negociación y comparación Humano vs IA

Etapa 1 (`etapa1.ipynb`) perfiló los 100 audios en crudo y calibró/validó el detector de
pitidos de censura. Etapa 2 (`etapa2.ipynb`) transcribió los 100 audios completos con
ElevenLabs (el motor que ganó el benchmark A/B de la Etapa 1). Este notebook construye,
sobre esas 100 transcripciones, las variables de negociación que responden las preguntas
de negocio definidas en la Etapa 0, y hace la comparación estadística Humano vs IA.

## Las 5 capas de análisis

Organizadas por prioridad — no todas se trabajan con el mismo rigor a propósito:

| # | Capa | Pregunta | Rigor | Requiere extracción nueva |
|---|---|---|---|---|
| 1 | Negocio | ¿La llamada consigue una gestión exitosa? | Completo | Sí (capa principal) |
| 2 | Negociación | ¿La IA convierte una oferta nueva en acuerdo, igual que el humano? | Completo | Sí (capa principal) |
| 3 | Objeciones | ¿Qué tipo de objeciones puede resolver cada agente? | Medio | Sí (capa principal) |
| 4 | Conducta conversacional | ¿Interrumpe, deja silencios, monopoliza? | Medio | No — reusa `actividad_voz.csv` de la Etapa 1 |
| 5 | Sentimiento | ¿Cambia el estado emocional del cliente? | Exploratorio | Sí (pasada liviana aparte) |


---
## 1. Cómo se construyeron las variables

Las variables de las capas 1, 2, 3 y 5 son de lenguaje libre: montos dichos de palabra,
un resultado que solo es legible en cómo termina la conversación, el tono del cliente.
A diferencia de los pitidos de censura de la Etapa 1 (una señal acústica con una regla
dura: tono de 1000 Hz, energía concentrada, amplitud plana, ≥100 ms), aquí no hay una
regla determinista razonable — la extracción es semántica.

**Esquema y prompt**: documentados como código en `src/extract_variables.py` — 24 campos
por llamada (tipo de gestión, monto, descuento, resultado, quién corta la llamada, por
qué, confianza de la extracción, etc.) más 4 campos de tono/empatía en una segunda
pasada, más liviana, para la capa 5.

**Cómo se ejecutó de verdad esta vez**: el plazo de entrega no daba tiempo para levantar
y probar un pipeline nuevo contra una API de LLM de texto — el `.env` del proyecto solo
tiene claves de ElevenLabs y Deepgram (motores de voz-a-texto), no de un LLM. La
extracción la hizo Claude Code leyendo directamente las 100 transcripciones, repartidas
en 4 lotes de 25 corridos en paralelo, aplicando el esquema de `extract_variables.py`
palabra por palabra. Esto es una desviación real de "código que cualquiera reejecuta con
un comando" — se documenta aquí en vez de esconderla; `extract_variables.py` deja listo
el mismo esquema y prompt para correrse contra una API real si alguien quiere
reproducirlo de forma 100% automática.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact, mannwhitneyu

gv = pd.read_csv('data/gestion_variables.csv')
print(gv.shape, "filas x columnas")
print(gv.grupo.value_counts().to_dict())
print()
print("Columnas:", list(gv.columns))

(100, 28) filas x columnas
{'humano': 50, 'ia': 50}

Columnas: ['file', 'grupo', 'tipo_gestion', 'contacto_efectivo', 'de_donde_llaman', 'origen_deuda', 'monto_total_actual_cop', 'descuento_pct', 'descuento_monto_cop', 'valor_a_pagar_cop', 'n_cuotas', 'fecha_max_pago', 'fecha_pago_acordada', 'comparte_medios_pago', 'cliente_puede_pagar', 'objecion_cliente', 'tipo_objecion', 'alternativa_propuesta', 'resultado_llamada', 'caso_exito', 'quien_corta_la_llamada', 'motivo_fin_llamada', 'confianza_extraccion', 'notas', 'tono_inicial_cliente', 'tono_final_cliente', 'cambio_tono', 'agente_se_adapta']


---
## 2. Validación manual

**Chequeo automático sobre los "casos de éxito":** si una llamada está marcada como
cierre pero el texto no tiene ningún turno atribuible al cliente, el "acuerdo" que
declara el agente no es verificable con el texto disponible (típicamente una falla de
diarización, no una falla de negociación). Se aplica sobre las filas ya corregidas — por
eso da 0 sospechosos: el hallazgo real (1 caso, `631a4d16`) ya está corregido abajo.


In [ ]:
import re, pathlib

EXITO = {'cierre_1ra_oferta', 'cierre_2da_oferta_o_alternativa', 'fecha_acordada_sin_descuento'}
cierres = gv[gv.resultado_llamada.isin(EXITO)]

sospechosos = []
for _, r in cierres.iterrows():
    p = pathlib.Path(f"salidas/transcripciones/{r.grupo}/{r.file}.txt")
    hablantes = set(re.findall(r'\] (speaker_\d+):', p.read_text(encoding='utf-8')))
    if len(hablantes) < 2:
        sospechosos.append(r.file)

print(f"{len(cierres)} filas marcadas como exito.")
print(f"{len(sospechosos)} con menos de 2 hablantes distintos en el texto: {sospechosos}")

38 filas marcadas como exito.
0 con menos de 2 hablantes distintos en el texto: []


**Muestra revisada a mano:** 25 de las 100 filas (todas las de `confianza_extraccion:
baja` + una muestra aleatoria de 15 más), releyendo cada `.txt` contra el valor extraído.
23/25 (92%) correctas al primer intento. Se encontraron y corrigieron 2 errores:

- **`2a92a64c`** (humano): el monto de deuda extraído originalmente (47.113.426) era la
  oferta *combinada* de 2 obligaciones que la clienta rechazó por desconfianza (*"me llama
  el uno, me llama el otro, ya no sé ni a quién creerle"*). El acuerdo que sí se cerró fue
  solo sobre una tarjeta de crédito, saldo real 1.148.855, pagado en 2 cuotas de 320.000.
  Corregido en `data/gestion_variables.csv`.
- **`631a4d16`** (IA): estaba marcado como cierre exitoso (`cierre_1ra_oferta`), pero es
  justamente el caso que detecta el chequeo automático de arriba — 0 turnos de cliente
  visibles en el texto, pese a que el bot declara *"me alegra que lleguemos a este
  acuerdo de pago"*. Reclasificado como `otro` / no verificable, para ser consistente con
  el mismo problema en `355812a5` (que sí se había clasificado bien desde el principio).

`caso_exito` se recalcula siempre de forma determinista a partir de `resultado_llamada`
(no se confía en el booleano que puso cada lote de extracción), para que sea auditable
desde una sola columna categórica.


### 2.1 Cómo se sabe quién es el agente y quién el cliente

ElevenLabs no distingue roles: `diarize=true` (ver `transcribe.py`) separa voces por
acústica y las etiqueta genéricamente `speaker_0`, `speaker_1` (a veces `speaker_2` si
hay un tercero) — no hay ningún campo de "agente"/"cliente" en la transcripción cruda.
Esa atribución la hace quien lee el texto (Claude Code, en esta extracción), por
contenido: quien se presenta, informa el monto de la deuda y ofrece la propuesta es el
agente; quien responde sobre su situación y acepta o rechaza es el cliente. Es un juicio
que se repite en cada archivo, no un dato estructurado — y hereda cualquier error de la
diarización (dos voces solapadas que quedan como un solo `speaker_id`, o al revés), tal
como advirtió la Etapa 1. No se validó por separado la atribución de rol en sí; la
validación manual de la sección 2 confirma los *valores* extraídos, asumiendo que el rol
ya estaba bien identificado.

### 2.2 Correcciones post-entrega, a partir de revisión del usuario

**`tipo_gestion`: se fusionan `seguimiento_acuerdo` y `renegociacion_fecha`.** La frontera
entre "llamada de seguimiento de un acuerdo ya pactado" y "renegociación de la fecha de
ese acuerdo" no se sostiene desde el texto: `77d258d0` quedó como `seguimiento_acuerdo`
pero tiene la misma firma que el bloque de `renegociacion_fecha` (objeción "no tiene
plata" + la fecha se mueve del 31 de julio al 15 de agosto). No es un caso raro, es la
prueba de que la distinción es artificial. Se fusiona en `seguimiento_o_renegociacion` —
esto no cambia ninguna cifra ya reportada: el corte usado para comparar contra IA siempre
fue `nueva_oferta` vs. todo lo demás, nunca se separaron estos dos grupos en ningún test.

**`de_donde_llaman`: se completa con regex para IA.** La extracción semántica original
dejó en `null` 18 llamadas de IA donde la frase de apertura del guion — *"del área de
embargos, judicializaciones y alivios financieros"* — está clarísima y sin censurar en
el texto (ver ejemplo abajo). Como esa frase es casi fija, se recupera con una regla
determinista en vez de una segunda pasada semántica.


In [ ]:
import re, glob

patron = re.compile(r'[Áá]rea de embargos[^.?]*', re.IGNORECASE)
archivos = glob.glob('salidas/transcripciones/ia/*.txt')
con_frase = sum(1 for f in archivos if patron.search(open(f, encoding='utf-8').read()))
print(f"{con_frase}/{len(archivos)} audios de IA dicen la frase de apertura del guion (regex).")

gv2 = pd.read_csv('data/gestion_variables.csv')
lleno_ia = gv2[gv2.grupo == 'ia'].de_donde_llaman.notna().sum()
lleno_humano = gv2[gv2.grupo == 'humano'].de_donde_llaman.notna().sum()
print(f"de_donde_llaman lleno tras la correccion -> ia: {lleno_ia}/50   humano: {lleno_humano}/50")
print("(en humano el hueco es real, no un error de extraccion: el nombre de la entidad")
print(" cae justo dentro del [CENSURADO] con mas frecuencia que la frase generica de la IA,")
print(" verificado releyendo 2 casos nulos -- 0445c357 y 09115a4a -- contra el .txt original)")

40/50 audios de IA dicen la frase de apertura del guion (regex).
de_donde_llaman lleno tras la correccion -> ia: 40/50   humano: 26/50
(en humano el hueco es real, no un error de extraccion: el nombre de la entidad
 cae justo dentro del [CENSURADO] con mas frecuencia que la frase generica de la IA,
 verificado releyendo 2 casos nulos -- 0445c357 y 09115a4a -- contra el .txt original)


---
## 3. Capa 1 — Negocio: ¿la llamada consigue una gestión exitosa?

`caso_exito = True` si `resultado_llamada` es `cierre_1ra_oferta`,
`cierre_2da_oferta_o_alternativa` o `fecha_acordada_sin_descuento` — los 3 casos de éxito
que se definieron en la Etapa 0 (acepta primera oferta / acepta tras negociar una
alternativa o acuerdo de largo plazo / al menos queda una fecha de pago acordada).


In [ ]:
def tasa_exito(df):
    ex = df.caso_exito.sum()
    n = len(df)
    return ex, n, ex / n * 100 if n else float('nan')

print("=== Tasa de exito GENERAL (sin filtrar tipo de gestion) ===")
tabla = {}
for g in ['humano', 'ia']:
    sub = gv[gv.grupo == g]
    ex, n, pct = tasa_exito(sub)
    tabla[g] = (ex, n - ex)
    print(f"  {g}: {ex}/{n} = {pct:.1f}%")
odds, p = fisher_exact([tabla['humano'], tabla['ia']])
print(f"  Fisher exact (humano vs ia): OR={odds:.2f}  p={p:.4f}")

print()
print("=== Tasa de exito SOLO tipo_gestion=nueva_oferta (comparacion justa vs IA) ===")
tabla2 = {}
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & (gv.tipo_gestion == 'nueva_oferta')]
    ex, n, pct = tasa_exito(sub)
    tabla2[g] = (ex, n - ex)
    print(f"  {g} (n={n}): {ex}/{n} = {pct:.1f}%")
odds2, p2 = fisher_exact([tabla2['humano'], tabla2['ia']])
print(f"  Fisher exact: OR={odds2:.2f}  p={p2:.4f}  <- deja de ser significativo (n chico)")

print()
print("=== Composicion de tipo_gestion por grupo ===")
print(pd.crosstab(gv.grupo, gv.tipo_gestion))

print()
print("=== Contacto efectivo (se logra hablar con el titular o quien decide) ===")
for g in ['humano', 'ia']:
    sub = gv[gv.grupo == g]
    ce = sub.contacto_efectivo.sum()
    print(f"  {g}: {ce}/{len(sub)} = {ce/len(sub)*100:.1f}%")

print()
print("=== Quien corta la llamada ===")
print(pd.crosstab(gv.grupo, gv.quien_corta_la_llamada))

print()
print("=== De las que se cortan tecnicamente, ¿ya habia contacto real? (solo IA) ===")
caidas = gv[(gv.grupo == 'ia') & (gv.quien_corta_la_llamada == 'se_corta_tecnicamente')]
print(f"  {len(caidas)} llamadas IA se cortan tecnicamente; {caidas.contacto_efectivo.sum()} de esas ya tenian contacto real")
con_monto = caidas[caidas.monto_total_actual_cop.notna()]
print(f"  {len(con_monto)} de esas mencionaron un monto de deuda antes de cortarse; suma = {con_monto.monto_total_actual_cop.sum():,.0f} COP")

=== Tasa de exito GENERAL (sin filtrar tipo de gestion) ===
  humano: 27/50 = 54.0%
  ia: 11/50 = 22.0%
  Fisher exact (humano vs ia): OR=4.16  p=0.0018

=== Tasa de exito SOLO tipo_gestion=nueva_oferta (comparacion justa vs IA) ===
  humano (n=18): 8/18 = 44.4%
  ia (n=37): 11/37 = 29.7%
  Fisher exact: OR=1.89  p=0.3677  <- deja de ser significativo (n chico)

=== Composicion de tipo_gestion por grupo ===
tipo_gestion  nueva_oferta  otro  seguimiento_o_renegociacion
grupo                                                        
humano                  18     5                           27
ia                      37    13                            0

=== Contacto efectivo (se logra hablar con el titular o quien decide) ===
  humano: 46/50 = 92.0%
  ia: 34/50 = 68.0%

=== Quien corta la llamada ===
quien_corta_la_llamada  agente  cliente  no_aplica  se_corta_tecnicamente
grupo                                                                    
humano                      41        3   

**Lectura:** la brecha cruda (54.0% vs 22.0%, p=0.0018) se reduce a 44.4% vs 29.7%
(p=0.37, ya no significativa) apenas se compara el mismo tipo de gestión en los dos
grupos — la IA hace casi exclusivamente `nueva_oferta` (37/50), mientras el humano
reparte su tiempo entre negociación nueva, seguimiento de acuerdos y renegociación de
fechas. Buena parte de la brecha cruda es composición de muestra, no habilidad de
negociación. Lo que sí es un hallazgo limpio, sin ambigüedad de comparabilidad: **1 de
cada 4 llamadas de IA (13/50) se pierde por falla técnica**, contra 1 de cada 50 en
humano — y en 10 de esas 13 ya había contacto real con el cliente, con ~$22.9M COP de
deuda mencionada antes del corte.


---
## 4. Capa 2 — Negociación: ¿con qué propuesta económica se consigue el resultado?

Se restringe a `tipo_gestion == 'nueva_oferta'` (la comparación directa: ambos grupos
ofreciendo un descuento nuevo sobre la deuda) y a las llamadas donde el acuerdo se cerró
(`caso_exito == True`), para comparar cuánto de la deuda original terminó pagando el
cliente.


In [ ]:
print("=== Descuento ofrecido (todas las llamadas nueva_oferta con el dato) ===")
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & (gv.tipo_gestion == 'nueva_oferta') & gv.descuento_pct.notna()]
    if len(sub):
        print(f"  {g}: n={len(sub)}  mediana={sub.descuento_pct.median():.1f}%  rango=[{sub.descuento_pct.min():.0f}, {sub.descuento_pct.max():.0f}]")

print()
print("=== Valor pactado / deuda total, SOLO nueva_oferta y caso_exito=True ===")
res = {}
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & (gv.tipo_gestion == 'nueva_oferta') & gv.caso_exito
             & gv.monto_total_actual_cop.notna() & gv.valor_a_pagar_cop.notna()
             & (gv.monto_total_actual_cop > 0)]
    ratios = (sub.valor_a_pagar_cop / sub.monto_total_actual_cop).tolist()
    res[g] = ratios
    print(f"  {g}: n={len(ratios)}  mediana={np.median(ratios)*100:.1f}%  media={np.mean(ratios)*100:.1f}%")
if res['humano'] and res['ia']:
    u, p = mannwhitneyu(res['humano'], res['ia'], alternative='two-sided')
    print(f"  Mann-Whitney U={u}  p={p:.4f}  <- sin diferencia significativa, n chico (8 vs 10)")

print()
print("=== Monto total de la deuda en juego (todas las filas con el dato) ===")
for g in ['humano', 'ia']:
    m = gv[(gv.grupo == g) & gv.monto_total_actual_cop.notna()].monto_total_actual_cop
    print(f"  {g}: n={len(m)}  mediana={m.median():,.0f}  min={m.min():,.0f}  max={m.max():,.0f} COP")

=== Descuento ofrecido (todas las llamadas nueva_oferta con el dato) ===
  humano: n=5  mediana=20.0%  rango=[10, 30]
  ia: n=31  mediana=7.0%  rango=[0, 47]

=== Valor pactado / deuda total, SOLO nueva_oferta y caso_exito=True ===
  humano: n=8  mediana=95.8%  media=87.8%
  ia: n=10  mediana=86.7%  media=85.1%
  Mann-Whitney U=48.0  p=0.4973  <- sin diferencia significativa, n chico (8 vs 10)

=== Monto total de la deuda en juego (todas las filas con el dato) ===
  humano: n=33  mediana=1,220,000  min=60,000  max=19,620,732 COP
  ia: n=33  mediana=1,820,000  min=70,000  max=25,880,000 COP


**Lectura:** ninguna diferencia detectable en cuánto "deja sobre la mesa" cada
agente cuando sí cierra un acuerdo (95.8% vs 86.7% de la deuda original, p=0.50) — con la
advertencia honesta de que son solo 8 y 10 casos. No hay evidencia de que la IA negocie
descuentos más generosos ni más tacaños que el humano; tampoco de lo contrario.


---
## 5. Capa 3 — Objeciones: ¿qué tipo de objeciones puede resolver cada agente?


In [ ]:
print("=== Tipos de objecion mas frecuentes por grupo ===")
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & gv.objecion_cliente & gv.tipo_objecion.notna()]
    top = sub.tipo_objecion.str.split(r'[(/]').str[0].str.strip().value_counts().head(5)
    print(f"  {g}:")
    print(top.to_string())
    print()

print("=== ¿Tener objecion cambia la probabilidad de exito? ===")
for g in ['humano', 'ia']:
    sub = gv[gv.grupo == g]
    con = sub[sub.objecion_cliente]
    sin = sub[~sub.objecion_cliente]
    ex_con, n_con = con.caso_exito.sum(), len(con)
    ex_sin, n_sin = sin.caso_exito.sum(), len(sin)
    print(f"  {g}: con objecion {ex_con}/{n_con}={ex_con/n_con*100:.1f}%  |  sin objecion {ex_sin}/{n_sin}={ex_sin/n_sin*100:.1f}%")

print()
print("=== OJO: composicion de 'sin objecion' en IA (posible artefacto) ===")
sin_obj_ia = gv[(gv.grupo == 'ia') & (~gv.objecion_cliente)]
print(sin_obj_ia.resultado_llamada.value_counts().to_string())

=== Tipos de objecion mas frecuentes por grupo ===
  humano:
tipo_objecion
no_tiene_plata                             16
disputa_monto                               5
desconfia_de_la_llamada                     4
no_reconoce_deuda                           2
problemas_tecnicos_recibiendo_documento     1

  ia:
tipo_objecion
no_tiene_plata             8
no_reconoce_deuda          4
pide_mas_tiempo            3
disputa_monto              3
desconfia_de_la_llamada    3

=== ¿Tener objecion cambia la probabilidad de exito? ===
  humano: con objecion 15/32=46.9%  |  sin objecion 12/18=66.7%
  ia: con objecion 9/29=31.0%  |  sin objecion 2/21=9.5%

=== OJO: composicion de 'sin objecion' en IA (posible artefacto) ===
resultado_llamada
no_es_titular        9
llamada_cortada      7
otro                 3
cierre_1ra_oferta    2


**Lectura:** los tipos de objeción son parecidos en ambos grupos — "no tiene
plata" domina en los dos. La comparación cruda "con objeción cierra más que sin objeción"
en IA (31.0% vs 9.5%) **no se puede leer como que las objeciones ayudan a cerrar**: el
90% (19/21) de las llamadas IA "sin objeción" nunca llegó a una negociación real (no era
el titular, o se cortó antes) — es un artefacto de composición, no una señal de manejo de
objeciones. En humano sí se ve el patrón esperado (66.7% sin objeción vs 46.9% con
objeción), aunque tampoco es significativo con este n (p=0.24).


---
## 6. Capa 4 — Conducta conversacional: ¿interrumpe, deja silencios, monopoliza?

Sin extracción nueva: se reusa `data/actividad_voz.csv`, construido en la Etapa 1 sobre
los 100 audios crudos (turnos, pausas, solapamiento de voz por doble periodicidad) y
nunca cruzado antes con el resultado de la negociación.


In [ ]:
act = pd.read_csv('data/actividad_voz.csv')
print(act.shape, "filas x columnas —", act.grupo.value_counts().to_dict())
print()

campos = ['turnos_por_min', 'dur_turno_mediana_s', 'pausa_mediana_s', 'pct_solape_sobre_voz', 'pct_voz']
for campo in campos:
    h = act[act.grupo == 'humano'][campo]
    i = act[act.grupo == 'ia'][campo]
    u, p = mannwhitneyu(h, i, alternative='two-sided')
    print(f"  {campo:22s}  humano mediana={h.median():7.2f}   ia mediana={i.median():7.2f}   Mann-Whitney p={p:.4f}")

(100, 20) filas x columnas — {'humano': 50, 'ia': 50}

  turnos_por_min          humano mediana=  43.25   ia mediana=  38.63   Mann-Whitney p=0.0085
  dur_turno_mediana_s     humano mediana=   0.59   ia mediana=   0.78   Mann-Whitney p=0.0000
  pausa_mediana_s         humano mediana=   0.46   ia mediana=   0.47   Mann-Whitney p=0.2392
  pct_solape_sobre_voz    humano mediana=  11.38   ia mediana=   7.28   Mann-Whitney p=0.0000
  pct_voz                 humano mediana=  76.85   ia mediana=  67.95   Mann-Whitney p=0.0000


**Lectura:** diferencias fuertes y limpias, sin ningún artefacto de composición
(se calculan sobre los 100 audios crudos, éxito o no): los humanos hablan en turnos casi
la mitad de cortos (0.59s vs 0.78s mediana, p<0.0001), con más turnos por minuto (43.3 vs
38.6, p=0.008), más solapamiento — se interrumpen más (11.4% vs 7.3% de la voz, p<0.0001)
— y llenan más tiempo de la llamada hablando (76.9% vs 68.0%, p<0.0001). La IA suena a
guion leído en párrafos largos; el humano suena a conversación con idas y vueltas.


---
## 7. Capa 5 — Sentimiento (exploratorio)

Pasada liviana adicional (2 campos de tono + 1 de adaptación) sobre las mismas 100
transcripciones, con menor rigor a propósito — es la capa de menor prioridad del
análisis.


In [ ]:
print("=== Cambio de tono del cliente durante la llamada (solo con cliente real) ===")
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & gv.cambio_tono.notna() & (gv.cambio_tono != 'no_aplica')]
    print(f"  {g} (n={len(sub)}):")
    print("   ", sub.cambio_tono.value_counts().to_dict())

print()
print("=== ¿El agente se adapta a lo que dice el cliente? ===")
for g in ['humano', 'ia']:
    sub = gv[(gv.grupo == g) & gv.agente_se_adapta.isin([True, False])]
    ad = sub.agente_se_adapta.sum()
    print(f"  {g}: {ad}/{len(sub)} = {ad/len(sub)*100:.1f}% (n aplicable={len(sub)})")

=== Cambio de tono del cliente durante la llamada (solo con cliente real) ===
  humano (n=47):
    {'igual': 26, 'mejora': 17, 'empeora': 4}
  ia (n=35):
    {'igual': 18, 'empeora': 10, 'mejora': 7}

=== ¿El agente se adapta a lo que dice el cliente? ===
  humano: 28/41 = 68.3% (n aplicable=41)
  ia: 11/31 = 35.5% (n aplicable=31)


**Lectura, con cautela (esta capa es exploratoria):** el cliente empeora de tono en
IA casi 3 veces más, proporcionalmente, que en humano (28.6% vs 8.5% de las llamadas con
cliente real). El agente humano se adapta a lo que dice el cliente el doble de veces que
la IA (68.3% vs 35.5%) — consistente con el patrón de la Capa 4: guion fijo vs.
conversación adaptativa. No se encontró una relación limpia entre "se adapta" y "cierra"
en ninguno de los dos grupos — no se fuerza esa lectura.


---
## 8. Síntesis

1. La brecha cruda de cierre (54% humano vs 22% IA) se reduce a la mitad y deja de ser
   estadísticamente significativa (44% vs 30%, p=0.37) al comparar solo el mismo tipo de
   gestión — buena parte de la brecha reportada ingenuamente sería composición de
   muestra, no desempeño.
2. El hallazgo más limpio y accionable: **1 de cada 4 llamadas de IA se pierde por falla
   técnica** (13/50 vs 1/50 en humano), con ~$22.9M COP de deuda ya en conversación real
   perdidos solo por eso — no por rechazo del cliente ni por mal guion.
3. Ni el descuento ofrecido ni el valor final recuperado muestran diferencia
   significativa entre agentes cuando ambos cierran (p=0.50) — no hay evidencia de que
   uno deje más plata sobre la mesa que el otro.
4. La IA habla en turnos casi el doble de largos, interrumpe/solapa menos y deja más
   silencio en la llamada — conducta de guion leído, no de conversación adaptativa
   (p<0.0001 en las 4 métricas).
5. El agente humano se adapta explícitamente a lo que dice el cliente el doble de veces
   que la IA (68% vs 36%), y el tono del cliente empeora proporcionalmente 3 veces más
   seguido en las llamadas de IA.
